# Predicción horaria con torres semánticas GRU-D

Este notebook entrena cinco torres multivariables con atención temporal. Sus resúmenes se combinan con el contexto del paciente para predecir la sepsis por hora.

La configuración inicial usa 500 pacientes del conjunto A de entrenamiento y reserva el 20 % para validación. La prueba final está desactivada por defecto.

Antes de empezar, ejecuta `uv sync --extra model --group dev` en la raíz del proyecto, selecciona el kernel de `.venv` y comprueba que `cleaning.ipynb` haya exportado las particiones.

In [1]:
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from tensorflow import keras

from appr_project.data import (
    PatientHours,
    Standardization,
    WindowBatches,
    load_partition,
    split_patients,
)
from appr_project.evaluation import predict_hours, summarize_metrics
from appr_project.models.semantic_grud import SEMANTIC_GROUPS
from appr_project.training import Hyperparameters, train_experiment

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
DATA_DIR = ROOT / "data" / "processed"


## 1. Configuración del piloto

`source` admite `"A"`, `"B"` y `"A+B"`. Para usar todos los pacientes de entrenamiento, cambia `sample_patients` a `None`. Con `folds=None` se usa una sola partición; un entero como `5` activa la validación cruzada.

`trials=1` entrena solo la configuración base. Con `trials=4` se prueban esa configuración y tres alternativas de la búsqueda aleatoria. El costo de la búsqueda aumenta con los ensayos y los folds.


In [26]:
settings = dict(
    architecture="semantic_grud",
    source="A",
    sample_patients=None,
    validation_fraction=0.20,
    folds=3,
    trials=1,
    epochs=20,
    patience=4,
    seed=2026,
)

base = Hyperparameters(
    window=15,
    units=8,
    dense_units=32,
    dropout=0.20,
    learning_rate=0.001,
    batch_size=128,
)


## 2. Inspeccionar la partición y un batch

Se muestran los pacientes, sus horas y cuántos tienen alguna etiqueta positiva. En CV, la vista previa corresponde al primer fold. La muestra y la partición se reproducen al entrenar con la misma semilla.

Cada entrada temporal tiene dimensiones `[batch, horas, variables, 4]`. Los cuatro canales son el valor estandarizado, la máscara de medición, el tiempo desde la medición previa y su valor. La máscara `valid_hours` marca las horas reales de la ventana.


In [27]:
frame = load_partition(DATA_DIR, "train", settings["source"])

train_ids, val_ids = next(split_patients(
    frame,
    validation_fraction=settings["validation_fraction"],
    folds=settings["folds"],
    seed=settings["seed"],
    sample_patients=settings["sample_patients"],
))

train_preview = frame.loc[frame.patient_id.isin(train_ids)]
val_preview = frame.loc[frame.patient_id.isin(val_ids)]

summary = []
for name, partition in (("train", train_preview), ("validation", val_preview)):
    patient_labels = partition.groupby("patient_id").SepsisLabel.max()
    summary.append({
        "partición": name,
        "pacientes": len(patient_labels),
        "pacientes positivos": int(patient_labels.sum()),
        "horas": len(partition),
    })
display(pd.DataFrame(summary).set_index("partición"))


,pacientes,pacientes positivos,horas
partición,,,
train,10846,955,418590
validation,5423,477,212441


In [28]:
scaler = Standardization.fit(train_preview)
first_patient = train_preview.loc[train_preview.patient_id.eq(train_ids[0])]
hours = PatientHours(first_patient, scaler)
batches = WindowBatches(hours, base.window, base.batch_size)
inputs, labels = batches[0]

print("Formas de entrada:", {name: array.shape for name, array in inputs.items()})
print("Forma de las etiquetas:", labels.shape)
print("Paciente:", hours.metadata["patient_id"].iloc[0])
print("Horas válidas de la primera ventana:", inputs["valid_hours"][0].astype(int).tolist())
print("Etiqueta de la primera ventana:", int(labels[0, 0]))

for tower, variables in SEMANTIC_GROUPS.items():
    print(tower, list(variables))


Formas de entrada: {'temporal': (54, 15, 26, 4), 'valid_hours': (54, 15), 'static': (54, 10)}
Forma de las etiquetas: (54, 1)
Paciente: A_000001
Horas válidas de la primera ventana: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
Etiqueta de la primera ventana: 0
hemodynamics ['HR', 'SBP', 'MAP', 'DBP', 'Lactate']
respiration ['Resp', 'O2Sat', 'SaO2', 'FiO2', 'PaCO2', 'pH']
renal_chemistry ['BUN', 'Creatinine', 'Calcium', 'Magnesium', 'Phosphate', 'Potassium', 'Glucose']
hepatic ['AST', 'Alkalinephos', 'Bilirubin_total']
hematology ['Hct', 'Hgb', 'WBC', 'Platelets', 'Temp']


## 3. Entrenar y revisar los resultados

Cada ejecución crea una carpeta nueva en `runs/` con el modelo, la configuración, las particiones, las métricas y los tiempos. El prefijo `notebook-semantic-grud` identifica estas corridas.

Para comparar con `modeling.ipynb`, comprueba que las asignaciones de pacientes en `splits.csv` coincidan y usa el mismo origen de prueba. Una vez elegida la configuración de cada modelo, compara la precisión promedio (AP) y la utilidad en los archivos `test_<origen>_metrics.csv`.

In [29]:
run_dir = ROOT / "runs" / f"notebook-semantic-grud-{settings['source']}-{datetime.now():%Y%m%d-%H%M%S-%f}"
validation_metrics = train_experiment(frame, run_dir, base=base, **settings)

print(f"Resultados: {run_dir}")
display(validation_metrics.round(4))


Trial 1/1, fold 1/3: Hyperparameters(window=15, units=8, dense_units=32, dropout=0.2, learning_rate=0.001, batch_size=128)
Epoch 1/20
3271/3271 - 166s - 51ms/step - loss: 0.1023 - pr_auc: 0.0502 - val_loss: 0.0911 - val_pr_auc: 0.0904
Epoch 2/20
3271/3271 - 206s - 63ms/step - loss: 0.0915 - pr_auc: 0.0960 - val_loss: 0.0898 - val_pr_auc: 0.0980
Epoch 3/20
3271/3271 - 133s - 41ms/step - loss: 0.0893 - pr_auc: 0.1126 - val_loss: 0.0899 - val_pr_auc: 0.0950
Epoch 4/20
3271/3271 - 141s - 43ms/step - loss: 0.0872 - pr_auc: 0.1292 - val_loss: 0.0901 - val_pr_auc: 0.0930
Epoch 5/20
3271/3271 - 153s - 47ms/step - loss: 0.0856 - pr_auc: 0.1443 - val_loss: 0.0918 - val_pr_auc: 0.0969
Epoch 6/20
3271/3271 - 152s - 46ms/step - loss: 0.0839 - pr_auc: 0.1631 - val_loss: 0.0919 - val_pr_auc: 0.0992
Epoch 7/20
3271/3271 - 145s - 44ms/step - loss: 0.0825 - pr_auc: 0.1798 - val_loss: 0.0942 - val_pr_auc: 0.1000
Epoch 8/20
3271/3271 - 146s - 45ms/step - loss: 0.0809 - pr_auc: 0.1981 - val_loss: 0.0926 - 

,source,patients,hours,positive_hours,positive_hour_fraction,average_precision,auroc,utility,alert_fraction,recall_at_threshold,threshold
0,all,16269,631031,13701,0.0217,0.092,0.7864,0.3685,0.2003,0.6096,0.0309
1,A,16269,631031,13701,0.0217,0.092,0.7864,0.3685,0.2003,0.6096,0.0309


In [30]:
display(pd.read_csv(run_dir / "trials.csv").round(4))

fold_metrics = pd.read_csv(run_dir / "fold_metrics.csv")
timing_columns = [
    "trial", "fold", "train_windows", "validation_windows",
    "preparation_seconds", "epochs_run", "seconds_per_epoch",
]
display(fold_metrics.loc[fold_metrics.source.eq("all"), timing_columns].round(2))


,trial,window,units,dense_units,dropout,learning_rate,batch_size,mean_validation_ap,std_validation_ap
0,1,15,8,32,0.2,0.001,128,0.0956,0.0042


,trial,fold,train_windows,validation_windows,preparation_seconds,epochs_run,seconds_per_epoch
0,1,1,418590,212441,3.84,11,162.78
2,1,2,421986,209045,5.63,9,192.81
4,1,3,421486,209545,5.57,6,251.46


## 4. Evaluación final opcional

Activa esta celda cuando hayas elegido la configuración definitiva. La utilidad y la proporción de horas con alerta en test se calculan con el umbral guardado en la corrida. Los resultados se muestran y se guardan por origen y de forma agregada. Usa `positive_hour_fraction` como referencia para el AP.


In [ ]:
evaluate_test = False
test_source = "A+B"  # También admite "A" o "B".

if evaluate_test:
    config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
    saved_model = keras.models.load_model(run_dir / "best.keras", compile=False)
    saved_scaler = Standardization(**config["standardization"])
    test = load_partition(DATA_DIR, "test", test_source)
    test_batches = WindowBatches(
        PatientHours(test, saved_scaler),
        config["hyperparameters"]["window"],
        config["hyperparameters"].get("batch_size", config.get("batch_size")),
    )
    test_predictions = predict_hours(saved_model, test_batches)
    test_metrics = summarize_metrics(test_predictions, config["threshold"])
    test_predictions.to_csv(run_dir / f"test_{test_source}_predictions.csv", index=False)
    test_metrics.to_csv(run_dir / f"test_{test_source}_metrics.csv", index=False)
    display(test_metrics.round(4))
